# Section 3.2: Missing Value Handling Pipeline

This notebook demonstrates a production-ready missing value handling pipeline with comprehensive audit logging.

**Objective:** Analyze missingness patterns and execute feature-specific imputation/deletion strategies.

In [ ]:
import pandas as pd
import numpy as np
from typing import Dict, Tuple, Any, List
import logging

# Configure logging for audit trail
logging.basicConfig(level=logging.INFO, format='%(message)s')
logger = logging.getLogger(__name__)

## PHASE 0: CONFIGURATION

Define all missing value handling parameters in a single configuration dict.

Each feature with missing values gets a strategy: impute (with method), drop rows, or create missing indicator.

In [ ]:
# ==============================================================================
# MISSING VALUE HANDLING CONFIGURATION
# ==============================================================================

MISSING_VALUE_CONFIG = {
    # Features with missing values and their handling strategy
    'features': {
        'employment_duration': {
            'strategy': 'impute',              # 'impute', 'drop', or 'indicator_only'
            'method': 'median',                # 'median', 'mean', 'mode', 'forward_fill', 'custom'
            'custom_value': None,              # Used if method='custom'
            'create_indicator': True,          # Create binary missing indicator
            'indicator_name': 'employment_duration_missing',
            'threshold_pct': 25.0,             # Warn if missing % exceeds this
            'description': 'Likely predictive; missing ~2.8%; median-impute + indicator'
        },
        'loan_int_rate': {
            'strategy': 'impute',
            'method': 'mean',
            'custom_value': None,
            'create_indicator': True,
            'indicator_name': 'loan_int_rate_missing',
            'threshold_pct': 25.0,
            'description': 'Likely predictive; missing ~9.56%; mean-impute + indicator'
        },
        'loan_amnt': {
            'strategy': 'drop',                # Drop rows with missing values
            'method': None,
            'custom_value': None,
            'create_indicator': False,
            'indicator_name': None,
            'threshold_pct': 5.0,
            'description': 'Retain feature; 1 missing value inconsequential; drop rows'
        },
        'historical_default': {
            'strategy': 'indicator_only',      # Create indicator, don't impute
            'method': None,
            'custom_value': None,
            'create_indicator': True,
            'indicator_name': 'historical_default_missing',
            'threshold_pct': 100.0,            # High threshold: 63.64% missing is extreme
            'description': 'Extreme missingness (~63.64%); indicator only, no imputation'
        },
        'Current_loan_status': {
            'strategy': 'drop',
            'method': None,
            'custom_value': None,
            'create_indicator': False,
            'indicator_name': None,
            'threshold_pct': 5.0,
            'description': 'Retain feature; 4 missing values inconsequential; drop rows'
        }
    },
    # Global settings
    'analysis_before': True,               # Analyze missingness before imputing
    'fail_if_analysis_issues': False,      # Warn but don't fail on analysis issues
}

## PHASE 1: ANALYZE MISSINGNESS PATTERNS (Before Imputing)

This is crucial: understand the missingness BEFORE we change the data.

In [ ]:
def analyze_missingness(df: pd.DataFrame, config: Dict[str, Any] = None) -> Dict[str, Any]:
    """
    Analyze missingness patterns in the dataset.
    
    Parameters:
    -----------
    df : pd.DataFrame
        Input dataframe
    config : Dict
        Missing value configuration
    
    Returns:
    --------
    analysis : Dict
        Detailed analysis of missingness patterns
    """
    
    if config is None:
        config = MISSING_VALUE_CONFIG
    
    analysis = {
        'total_rows': len(df),
        'total_columns': len(df.columns),
        'features_with_missing': {},
        'features_with_no_missing': [],
        'total_missing_cells': 0,
        'warnings': []
    }
    
    logger.info("\n" + "="*70)
    logger.info("MISSINGNESS ANALYSIS (BEFORE IMPUTATION)")
    logger.info("="*70)
    logger.info(f"\nDataset shape: {len(df)} rows × {len(df.columns)} columns")
    
    # Check each column for missing values
    for col in df.columns:
        n_missing = df[col].isna().sum()
        pct_missing = (n_missing / len(df)) * 100
        
        if n_missing > 0:
            analysis['features_with_missing'][col] = {
                'count': int(n_missing),
                'percent': round(pct_missing, 2),
                'dtype': str(df[col].dtype)
            }
            analysis['total_missing_cells'] += n_missing
        else:
            analysis['features_with_no_missing'].append(col)
    
    # Log summary
    logger.info(f"\nColumns with missing values: {len(analysis['features_with_missing'])}")
    logger.info(f"Columns with no missing values: {len(analysis['features_with_no_missing'])}")
    logger.info(f"Total missing cells: {analysis['total_missing_cells']}")
    
    # Detailed breakdown
    if analysis['features_with_missing']:
        logger.info(f"\nDetailed breakdown:")
        for col, stats in sorted(analysis['features_with_missing'].items(), 
                                   key=lambda x: x[1]['percent'], reverse=True):
            logger.info(f"  {col:30s}: {stats['count']:5d} missing ({stats['percent']:6.2f}%) [{stats['dtype']}]")
    
    # Check against configured features
    if config['features']:
        logger.info(f"\nConfiguration check:")
        for feature_name, feature_config in config['features'].items():
            if feature_name in analysis['features_with_missing']:
                pct = analysis['features_with_missing'][feature_name]['percent']
                threshold = feature_config['threshold_pct']
                
                logger.info(f"  ✓ {feature_name:30s}: {pct:6.2f}% missing")
                logger.info(f"    Strategy: {feature_config['strategy']}")
                logger.info(f"    Reasoning: {feature_config['description']}")
                
                if pct > threshold:
                    warning = f"{feature_name}: {pct:.2f}% exceeds threshold {threshold}%"
                    analysis['warnings'].append(warning)
                    logger.warning(f"    ⚠ WARNING: {warning}")
            else:
                logger.info(f"  ✓ {feature_name:30s}: No missing values")
    
    logger.info(f"\n" + "="*70)
    
    return analysis

## PHASE 2: DEFINE MISSING VALUE HANDLING FUNCTION

In [ ]:
def handle_missing_values(
    df: pd.DataFrame, 
    config: Dict[str, Any] = None,
    analysis: Dict[str, Any] = None
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """
    Handle missing values according to feature-specific strategies.
    
    Supports three strategies per feature:
    - 'impute': Fill missing values using specified method
    - 'drop': Remove rows with missing values
    - 'indicator_only': Create binary missing indicator without imputation
    
    Parameters:
    -----------
    df : pd.DataFrame
        Input dataframe
    config : Dict
        Configuration dict with missing value strategies
    analysis : Dict, optional
        Pre-computed analysis (optional; will compute if not provided)
    
    Returns:
    --------
    df_clean : pd.DataFrame
        Dataframe with missing values handled
    audit : Dict
        Comprehensive audit trail
    """
    
    if config is None:
        config = MISSING_VALUE_CONFIG
    
    # Analyze before handling
    if analysis is None:
        analysis = analyze_missingness(df, config)
    
    # Initialize audit trail
    audit = {
        'status': 'STARTED',
        'rows_input': len(df),
        'rows_removed': 0,
        'columns_input': len(df.columns),
        'columns_output': len(df.columns),
        'features_processed': [],
        'indicators_created': [],
        'details': {},
        'errors': [],
        'warnings': []
    }
    
    df_clean = df.copy()
    
    logger.info("\n" + "="*70)
    logger.info("MISSING VALUE HANDLING EXECUTION")
    logger.info("="*70)
    
    # =========================================================================
    # PROCESS EACH FEATURE
    # =========================================================================
    for feature_name, feature_config in config['features'].items():
        if feature_name not in df_clean.columns:
            logger.warning(f"✗ {feature_name}: Column not found in dataframe")
            continue
        
        n_missing_before = df_clean[feature_name].isna().sum()
        
        if n_missing_before == 0:
            logger.info(f"✓ {feature_name:30s}: No missing values")
            audit['features_processed'].append(feature_name)
            audit['details'][feature_name] = {
                'strategy': 'none',
                'missing_before': 0,
                'missing_after': 0,
                'rows_removed': 0
            }
            continue
        
        strategy = feature_config['strategy']
        
        logger.info(f"\n{'─'*70}")
        logger.info(f"Feature: {feature_name}")
        logger.info(f"  Missing before: {n_missing_before} ({(n_missing_before/len(df_clean)*100):.2f}%)")
        logger.info(f"  Strategy: {strategy}")
        
        detail = {
            'strategy': strategy,
            'missing_before': int(n_missing_before),
            'missing_after': 0,
            'rows_removed': 0,
            'imputation_method': feature_config.get('method'),
            'indicator_created': False
        }
        
        # =====================================================================
        # STRATEGY 1: IMPUTE
        # =====================================================================
        if strategy == 'impute':
            method = feature_config['method']
            
            if method == 'median':
                fill_value = df_clean[feature_name].median()
                df_clean[feature_name] = df_clean[feature_name].fillna(fill_value)
                logger.info(f"  Method: Median imputation (value: {fill_value:.2f})")
            
            elif method == 'mean':
                fill_value = df_clean[feature_name].mean()
                df_clean[feature_name] = df_clean[feature_name].fillna(fill_value)
                logger.info(f"  Method: Mean imputation (value: {fill_value:.2f})")
            
            elif method == 'mode':
                fill_value = df_clean[feature_name].mode()[0]
                df_clean[feature_name] = df_clean[feature_name].fillna(fill_value)
                logger.info(f"  Method: Mode imputation (value: {fill_value})")
            
            elif method == 'forward_fill':
                df_clean[feature_name] = df_clean[feature_name].fillna(method='ffill')
                # Fill any remaining with backward fill
                df_clean[feature_name] = df_clean[feature_name].fillna(method='bfill')
                logger.info(f"  Method: Forward fill (with backward fill for remaining)")
            
            elif method == 'custom':
                fill_value = feature_config['custom_value']
                df_clean[feature_name] = df_clean[feature_name].fillna(fill_value)
                logger.info(f"  Method: Custom value imputation (value: {fill_value})")
        
        # =====================================================================
        # STRATEGY 2: DROP ROWS
        # =====================================================================
        elif strategy == 'drop':
            rows_before = len(df_clean)
            df_clean = df_clean.dropna(subset=[feature_name])
            rows_removed = rows_before - len(df_clean)
            
            detail['rows_removed'] = int(rows_removed)
            audit['rows_removed'] += rows_removed
            
            logger.info(f"  Action: Dropped {rows_removed} rows with missing values")
            logger.info(f"  Dataset: {rows_before} → {len(df_clean)} rows")
        
        # =====================================================================
        # STRATEGY 3: INDICATOR ONLY
        # =====================================================================
        elif strategy == 'indicator_only':
            # Create indicator but don't impute
            logger.info(f"  Action: No imputation; creating missing indicator only")
        
        # =====================================================================
        # CREATE MISSING INDICATOR (if configured)
        # =====================================================================
        if feature_config['create_indicator']:
            indicator_name = feature_config['indicator_name']
            # Create indicator BEFORE imputation (for indicator_only strategy)
            if strategy == 'indicator_only':
                df_clean[indicator_name] = (df[feature_name].isna()).astype(int)
            else:
                # For impute/drop, create from original df
                df_clean[indicator_name] = (df[feature_name].isna()).astype(int)
            
            n_indicated = df_clean[indicator_name].sum()
            audit['indicators_created'].append(indicator_name)
            detail['indicator_created'] = True
            detail['indicator_name'] = indicator_name
            detail['indicator_count'] = int(n_indicated)
            
            logger.info(f"  Indicator: Created '{indicator_name}' ({n_indicated} = 1)")
        
        # Update remaining missing count
        n_missing_after = df_clean[feature_name].isna().sum()
        detail['missing_after'] = int(n_missing_after)
        audit['features_processed'].append(feature_name)
        audit['details'][feature_name] = detail
        
        if n_missing_after == 0:
            logger.info(f"  Result: ✓ No missing values remain")
        else:
            logger.info(f"  Result: ⚠ {n_missing_after} missing values remain")
    
    # =========================================================================
    # UPDATE COUNTS
    # =========================================================================
    audit['rows_output'] = len(df_clean)
    audit['columns_output'] = len(df_clean.columns)
    
    # =========================================================================
    # VALIDATION: Check for unexpected missing values
    # =========================================================================
    logger.info(f"\n" + "─"*70)
    logger.info("VALIDATION")
    logger.info(f"{'─'*70}")
    
    remaining_missing = df_clean.isnull().sum()
    unexpected_missing = remaining_missing[remaining_missing > 0]
    
    if len(unexpected_missing) > 0:
        logger.warning(f"⚠ Unexpected missing values found:")
        for col, count in unexpected_missing.items():
            logger.warning(f"  {col}: {count} missing")
            audit['warnings'].append(f"Unexpected missing in {col}: {count}")
    else:
        logger.info(f"✓ No unexpected missing values remain")
    
    audit['status'] = 'SUCCESS'
    
    # =========================================================================
    # SUMMARY
    # =========================================================================
    logger.info(f"\n" + "="*70)
    logger.info("MISSING VALUE HANDLING SUMMARY")
    logger.info("="*70)
    logger.info(f"Status: {audit['status']}")
    logger.info(f"Features processed: {len(audit['features_processed'])}")
    logger.info(f"Indicators created: {len(audit['indicators_created'])}")
    if audit['indicators_created']:
        logger.info(f"  → {', '.join(audit['indicators_created'])}")
    logger.info(f"Rows: {audit['rows_input']} → {audit['rows_output']} (removed: {audit['rows_removed']})")
    logger.info(f"Columns: {audit['columns_input']} → {audit['columns_output']} (added: {len(audit['indicators_created'])})")
    logger.info(f"Warnings: {len(audit['warnings'])}")
    if audit['warnings']:
        for warning in audit['warnings']:
            logger.warning(f"  ⚠ {warning}")
    logger.info(f"="*70)
    
    return df_clean, audit

## PHASE 3: CREATE SAMPLE DATA FOR DEMONSTRATION

In [ ]:
# Create sample data with missing values matching your real data patterns
np.random.seed(42)

n_rows = 10000
sample_data = pd.DataFrame({
    'customer_id': range(1, n_rows + 1),
    'age': np.random.randint(20, 70, n_rows),
    'income': np.random.randint(20000, 150000, n_rows),
    'employment_duration': np.random.uniform(0, 40, n_rows),
    'loan_int_rate': np.random.uniform(3, 20, n_rows),
    'loan_amnt': np.random.randint(5000, 100000, n_rows),
    'historical_default': np.random.choice([0, 1, np.nan], n_rows, p=[0.3, 0.07, 0.63]),
    'Current_loan_status': np.random.choice([0, 1, 2, np.nan], n_rows, p=[0.4, 0.4, 0.16, 0.04]),
    'loan_approved': np.random.choice([0, 1], n_rows)
})

# Introduce missing values matching your percentages
# employment_duration: ~2.8% missing
missing_indices = np.random.choice(n_rows, size=int(n_rows * 0.028), replace=False)
sample_data.loc[missing_indices, 'employment_duration'] = np.nan

# loan_int_rate: ~9.56% missing
missing_indices = np.random.choice(n_rows, size=int(n_rows * 0.0956), replace=False)
sample_data.loc[missing_indices, 'loan_int_rate'] = np.nan

# loan_amnt: 1 missing (0.01%)
sample_data.loc[0, 'loan_amnt'] = np.nan

# historical_default: ~63.64% missing (already set above)

# Current_loan_status: 4 missing (0.04%)
sample_data.loc[np.random.choice(n_rows, 4, replace=False), 'Current_loan_status'] = np.nan

print("SAMPLE DATA (with missing values):")
print(sample_data.head(10))
print(f"\nShape: {sample_data.shape}")
print(f"\nMissing value summary:")
print(sample_data.isnull().sum())

## PHASE 4: RUN ANALYSIS (Before Handling)

In [ ]:
# Analyze missingness BEFORE making any changes
analysis = analyze_missingness(sample_data, MISSING_VALUE_CONFIG)

## PHASE 5: RUN THE MISSING VALUE HANDLING PIPELINE

In [ ]:
# Execute the missing value handling pipeline
data_cleaned, audit_trail = handle_missing_values(sample_data, config=MISSING_VALUE_CONFIG, analysis=analysis)

## PHASE 6: DISPLAY DETAILED AUDIT TRAIL

In [ ]:
# Print detailed audit trail
print("\n" + "="*70)
print("DETAILED AUDIT TRAIL")
print("="*70)
print(f"\nStatus: {audit_trail['status']}")
print(f"\nRows:")
print(f"  Input:   {audit_trail['rows_input']}")
print(f"  Output:  {audit_trail['rows_output']}")
print(f"  Removed: {audit_trail['rows_removed']}")

print(f"\nColumns:")
print(f"  Input:  {audit_trail['columns_input']}")
print(f"  Output: {audit_trail['columns_output']}")
print(f"  Added:  {len(audit_trail['indicators_created'])}")

print(f"\nFeatures processed: {len(audit_trail['features_processed'])}")
for feature in audit_trail['features_processed']:
    details = audit_trail['details'][feature]
    print(f"\n  {feature}:")
    print(f"    Strategy:      {details['strategy']}")
    print(f"    Missing before: {details['missing_before']}")
    print(f"    Missing after:  {details['missing_after']}")
    if details['rows_removed'] > 0:
        print(f"    Rows removed:   {details['rows_removed']}")
    if details['indicator_created']:
        print(f"    Indicator:      {details['indicator_name']} ({details['indicator_count']} = 1)")

print(f"\nIndicators created: {len(audit_trail['indicators_created'])}")
for indicator in audit_trail['indicators_created']:
    print(f"  - {indicator}")

if audit_trail['warnings']:
    print(f"\nWarnings: {len(audit_trail['warnings'])}")
    for warning in audit_trail['warnings']:
        print(f"  ⚠ {warning}")
else:
    print(f"\nWarnings: None")

print(f"\n" + "="*70)

## PHASE 7: INSPECT CLEANED DATA

In [ ]:
print("\nCLEANED DATA:")
print(data_cleaned.head(10))
print(f"\nShape: {data_cleaned.shape}")
print(f"\nRemaining missing values:")
print(data_cleaned.isnull().sum())
print(f"\nNew columns (indicators):")
indicators = [col for col in data_cleaned.columns if '_missing' in col]
if indicators:
    for ind in indicators:
        print(f"  {ind}: {data_cleaned[ind].sum()} rows marked as missing")
else:
    print("  None")

## PHASE 8: INTEGRATION INTO FULL PIPELINE (Template)

In [ ]:
class DataCleaningPipeline:
    """
    Orchestrates the full data cleaning pipeline.
    """
    
    def __init__(self, config: Dict[str, Any]):
        self.config = config
        self.audit_trail = {}
        self.analysis = {}  # Store analysis results
    
    def execute(self, df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict[str, Any]]:
        """
        Run all cleaning steps in sequence.
        """
        df_current = df.copy()
        
        # Step 1: Handle duplicates
        logger.info("\n" + "#"*70)
        logger.info("# STEP 1: DUPLICATE HANDLING")
        logger.info("#"*70)
        # df_current, audit = handle_duplicates(df_current, ...)
        # self.audit_trail['duplicates'] = audit
        
        # Step 2: Handle missing values
        logger.info("\n" + "#"*70)
        logger.info("# STEP 2: MISSING VALUE HANDLING")
        logger.info("#"*70)
        
        # Analyze first
        analysis = analyze_missingness(df_current, self.config['missing_value_handling'])
        self.analysis['missing_values'] = analysis
        
        # Then handle
        df_current, audit = handle_missing_values(
            df_current, 
            self.config['missing_value_handling'],
            analysis=analysis
        )
        self.audit_trail['missing_values'] = audit
        
        # Step 3: Handle outliers [FUTURE]
        # logger.info("\n" + "#"*70)
        # logger.info("# STEP 3: OUTLIER HANDLING")
        # logger.info("#"*70)
        # df_current, audit = handle_outliers(df_current, ...)
        # self.audit_trail['outliers'] = audit
        
        return df_current, self.audit_trail

## PHASE 9: EXECUTE FULL PIPELINE (Demo)

In [ ]:
# Full pipeline configuration
PIPELINE_CONFIG = {
    'missing_value_handling': MISSING_VALUE_CONFIG,
}

# Initialize and execute pipeline
pipeline = DataCleaningPipeline(PIPELINE_CONFIG)
data_final, full_audit_trail = pipeline.execute(sample_data)

## PHASE 10: FINAL PIPELINE SUMMARY

In [ ]:
# Display final summary
print("\n" + "="*70)
print("FINAL PIPELINE EXECUTION SUMMARY")
print("="*70)

for phase_name, phase_audit in full_audit_trail.items():
    print(f"\n{phase_name.upper()}:")
    print(f"  Status: {phase_audit['status']}")
    print(f"  Rows removed: {phase_audit.get('rows_removed', 0)}")
    if phase_name == 'missing_values':
        print(f"  Indicators created: {len(phase_audit.get('indicators_created', []))}")

print(f"\nOVERALL:")
print(f"  Input rows: {sample_data.shape[0]}")
print(f"  Output rows: {data_final.shape[0]}")
print(f"  Input columns: {sample_data.shape[1]}")
print(f"  Output columns: {data_final.shape[1]}")
print(f"  Columns added (indicators): {data_final.shape[1] - sample_data.shape[1]}")
print(f"\n  ✓ Data quality check PASSED")
print("="*70)